# Task 2: Move Prediction (highlighted last move)

Chessboard understanding with Qwen3.5-0.8B and token-level patch reordering (REOrder). Train on Colab; every training run and evaluation is cached on the Hugging Face Hub, so re-running a cell skips work already done.

Downloading Libraries and Imports

In [17]:

!pip install -q --upgrade "pillow<11.0.0" "transformers>=4.45.0" accelerate scikit-learn tqdm chess cairosvg python-Levenshtein datasets peft huggingface_hub python-dotenv
!pip uninstall -y torchao -q
!pip install -q flash-linear-attention

import gc
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import login
from peft import LoraConfig, PeftModel, TaskType
from transformers import AutoModelForImageTextToText, AutoProcessor


Setting up environment

In [18]:
# Project configuration dictionary
CONFIG = {
    "colab": True,
    "branch": "main",
    "repo_name": "BigDataAndTextMiningProject",
    "repo_owner": "Aivon99",
    "repo_dir": "/content/BigDataAndTextMiningProject",
}

# Clone the repository on Colab, or use the local checkout
repo_root = Path(CONFIG["repo_dir"])

if CONFIG["colab"]:
    if repo_root.exists():
        # /content survives a session restart, so update the existing clone
        print(f"Repository directory already exists at: {repo_root}; pulling latest changes...")
        pull = subprocess.run(["git", "-C", str(repo_root), "pull", "--ff-only"], capture_output=True, text=True)
        print((pull.stdout or pull.stderr).strip())
    else:
        auth_url = "https://"
        repo_url = f"{auth_url}github.com/{CONFIG['repo_owner']}/{CONFIG['repo_name']}.git"

        print(f"Cloning repository from {repo_url}...")
        result = subprocess.run(
            ["git", "clone", "--single-branch", "--branch", CONFIG["branch"], repo_url, str(repo_root)],
            capture_output=True, text=True
        )
        assert result.returncode == 0, f"Git clone failed: {result.stderr}"
else:
    repo_root = Path(".").resolve().parent


print(f"Setup Complete. REPO_ROOT: {repo_root}")

# Check GPU and device availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Make the project modules importable
sys.path.insert(0, str(repo_root / "src"))

from eval.results import (
    load_or_init_results_table, append_or_replace, load_results_table, push_results_table, evaluate_with_order,
)
from eval.diagnostics import (
    headline_table, analyze_san_predictions, san_error_breakdown, training_health_report,
    backfill_cached_metrics, save_diagnostics,
)
from training.finetune import finetune_and_push_chessboard_model, experiment_repo_id, experiment_model_name
from training.learned_reordering import train_with_learned_reordering, describe_order, greedy_order
from training.reordering import resolve_square_order
from training.resolution import configure_image_resolution
from training.token_reordering import verify_token_reordering

# Hub namespace of every model, dataset and results table of the project
hf_org_prefix = "bdatm-project"

print("All custom modules and eval utilities imported successfully!")

Repository directory already exists at: /content/BigDataAndTextMiningProject; pulling latest changes...
Already up to date.
Setup Complete. REPO_ROOT: /content/BigDataAndTextMiningProject
Using device: cuda
GPU: NVIDIA L4
All custom modules and eval utilities imported successfully!


Downloading dataset for task 2 from HuggingFace repo

In [19]:
# Authenticate with Hugging Face:
# - Colab: reads a token from a Colab secret named HF_TOKEN (key icon, left sidebar)
# - Local: reads HF_TOKEN from a repo-root .env file, or the environment
# - Falls back to an interactive login prompt if neither is found
hf_token = None
if CONFIG["colab"]:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None
else:
    from dotenv import load_dotenv
    load_dotenv(repo_root / ".env")
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Logged in to Hugging Face Hub using HF_TOKEN.")
else:
    print("No HF_TOKEN found in secrets/.env/environment, falling back to interactive login.")
    login()

dataset_name = "bdatm-project/dataset_task2"
print(f"Downloading dataset '{dataset_name}'...")

dataset_task2 = load_dataset(dataset_name, num_proc=16)

print("\nDataset loaded successfully!")
print(dataset_task2)
print("\nStructure sample of train split:")
print(dataset_task2["train"][0])

Logged in to Hugging Face Hub using HF_TOKEN.


Resolving data files:   0%|          | 0/3201 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/401 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/401 [00:00<?, ?it/s]


Dataset loaded successfully!
DatasetDict({
    train: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image', 'file_name_t1'],
        num_rows: 3200
    })
    validation: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image', 'file_name_t1'],
        num_rows: 400
    })
    test: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image', 'file_name_t1'],
        num_rows: 400
    })
})

Structure sample of train split:
{'sample_id': 'sample_000000', 'puzzle_id': 'drw7g', 'task': 'task2', 'fen': 'r3k2r/pppq2pp/2nb1p2/1B1Q3P/8/1P2P2P/PBPP1P2/R3K2R w KQkq - 1 13', 'prompt': 'You are a specialized model for chessboard understanding.\nYour goal is to predict the last move played on the board based on the highlighted squares.\nInput:\n- Board Image: The visual representation with the last move highlighted.\nOutput Format:\nReturn only the move in Standard Algebra

## Experiment setup

**Token-level patch reordering (REOrder).** A scan order is applied to the visual *tokens* inside the model: each image's token embeddings and their M-RoPE positions are permuted together (`src/training/token_reordering.py`), so the board image is never altered and every token keeps its true 2D position. Only the order in which the LLM reads them changes, which matters here because 18 of Qwen3.5's 24 LLM layers are Gated DeltaNet (a recurrent scan with no positional encoding).

The first runs of this notebook rearranged *pixel tiles* instead, which changed the board itself (a "file_wise" image was a transposed board). Their results stay in the results table, labelled `legacy: pixel-tile reordering, 512px`.

**Resolution (patch alignment).** At `IMAGE_SIZE = 256` the LLM sees an 8x8 token grid: exactly one token per chess square. The legacy 512px raster run is the resolution ablation (512px = 2x2 tokens per square).

In [20]:
IMAGE_SIZE = 256   # 256 -> 1 LLM token per square; 512 -> 2x2 tokens per square
TAG = f"tok{IMAGE_SIZE}"   # suffix of every Hub repo / results row of this setup
# Fixed scan orders compared with the raster baseline ("rank_wise" is raster itself).
# "hilbert" is one of REOrder's own orders.
STRATEGIES = ["zigzag", "spiral_center_out", "file_wise", "hilbert"]
N_SHOT = 3   # few-shot examples for the vanilla model, drawn from train
TASK = "task2"
model_id = "Qwen/Qwen3.5-0.8B"
hf_org_prefix = "bdatm-project"

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)


def load_base_model():
    return AutoModelForImageTextToText.from_pretrained(
        model_id, dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32, device_map="auto"
    )


def free_gpu():
    """Call after `del`-ing the model variables, so their GPU memory is released."""
    gc.collect()
    torch.cuda.empty_cache()


for s in ["raster", *STRATEGIES]:
    print(f"{s} (reading step of each square):\n{describe_order(resolve_square_order(s))}\n")

raster (reading step of each square):
8  0  1  2  3  4  5  6  7
7  8  9 10 11 12 13 14 15
6 16 17 18 19 20 21 22 23
5 24 25 26 27 28 29 30 31
4 32 33 34 35 36 37 38 39
3 40 41 42 43 44 45 46 47
2 48 49 50 51 52 53 54 55
1 56 57 58 59 60 61 62 63
   a  b  c  d  e  f  g  h

zigzag (reading step of each square):
8  0  1  2  3  4  5  6  7
7 15 14 13 12 11 10  9  8
6 16 17 18 19 20 21 22 23
5 31 30 29 28 27 26 25 24
4 32 33 34 35 36 37 38 39
3 47 46 45 44 43 42 41 40
2 48 49 50 51 52 53 54 55
1 63 62 61 60 59 58 57 56
   a  b  c  d  e  f  g  h

spiral_center_out (reading step of each square):
8 63 62 61 60 59 58 57 56
7 36 35 34 33 32 31 30 55
6 37 16 15 14 13 12 29 54
5 38 17  4  3  2 11 28 53
4 39 18  5  0  1 10 27 52
3 40 19  6  7  8  9 26 51
2 41 20 21 22 23 24 25 50
1 42 43 44 45 46 47 48 49
   a  b  c  d  e  f  g  h

file_wise (reading step of each square):
8  0  8 16 24 32 40 48 56
7  1  9 17 25 33 41 49 57
6  2 10 18 26 34 42 50 58
5  3 11 19 27 35 43 51 59
4  4 12 20 28 36 44 52 60

## Vanilla model

The processor is set to `IMAGE_SIZE` once here; training, evaluation and generation all go through it. `verify_token_reordering` checks on this environment's `transformers` version that no-order output equals the unpatched model, that embeddings and positions move together, and that the order actually changes the output.

In [21]:
model_vanilla = load_base_model()
processor = AutoProcessor.from_pretrained(model_id)
configure_image_resolution(processor, IMAGE_SIZE)
verify_token_reordering(model_vanilla, processor, dataset_task2["test"][0]["image"], resolve_square_order("zigzag"))

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

[resolution] processor set to 256px: {'image_size': 256, 'patch_grid': (16, 16), 'llm_token_grid': (8, 8), 'tokens_per_square': 1}
[token reordering] OK: {'image_tokens': 64, 'token_grid': (8, 8), 'max_logit_change': 2.0}


{'image_tokens': 64, 'token_grid': (8, 8), 'max_logit_change': 2.0}

In [22]:
# Grab the first test sample for Task 2
test_sample = dataset_task2["test"][0]

ground_truth_move = test_sample["target"]
task_prompt = test_sample["prompt"]
sample_id = test_sample["sample_id"]
board_image = test_sample["image"]

print(f"Sample ID: {sample_id}")
print(f"Prompt provided to the model:\n{task_prompt}\n")
print(f"Real Move (Ground Truth SAN): {ground_truth_move}\n")

# Prepare the multimodal input format for the model
chat_messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": board_image},
            {"type": "text", "text": task_prompt},
        ]
    }
]

# Apply the processor's chat template
formatted_text = processor.apply_chat_template(chat_messages, tokenize=False, add_generation_prompt=True)

# Tokenize inputs and move them to the GPU device
model_inputs = processor(
    text=[formatted_text],
    images=board_image,
    padding=True,
    return_tensors="pt"
).to(model_vanilla.device)

# Generate the zero-shot prediction
print("Generating zero-shot prediction for Task 2...")
with torch.no_grad():
    # SAN notation is short, so we can use a smaller max_new_tokens value
    output_token_ids = model_vanilla.generate(**model_inputs, max_new_tokens=16)

# Trim prompt tokens from the generated output
trimmed_output_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, output_token_ids)
]
predicted_move = processor.batch_decode(
    trimmed_output_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]

print(f"Predicted Move (Zero-Shot): {predicted_move.strip()}")

Sample ID: sample_000000
Prompt provided to the model:
You are a specialized model for chessboard understanding.
Your goal is to predict the last move played on the board based on the highlighted squares.
Input:
- Board Image: The visual representation with the last move highlighted.
Output Format:
Return only the move in Standard Algebraic Notation (SAN).

Real Move (Ground Truth SAN): Bxe4

Generating zero-shot prediction for Task 2...
Predicted Move (Zero-Shot): The move is 1. c4 (White)


Zero-shot and few-shot evaluation of the vanilla model. Few-shot prepends `N_SHOT` solved train examples as earlier chat turns.

In [23]:
all_models_results = load_or_init_results_table(task=TASK, hf_org_prefix=hf_org_prefix)

vanilla_results_df, vanilla_summary_df = evaluate_with_order(
    TASK, model_vanilla, processor, dataset_task2["test"], hf_org_prefix=hf_org_prefix,
    model_name=experiment_model_name("Vanilla Qwen3.5-0.8B (Zero-Shot)", TAG),
)
all_models_results = append_or_replace(all_models_results, vanilla_summary_df)

few_shot_examples = [dataset_task2["train"][i] for i in range(N_SHOT)]
fewshot_results_df, fewshot_summary_df = evaluate_with_order(
    TASK, model_vanilla, processor, dataset_task2["test"], hf_org_prefix=hf_org_prefix,
    model_name=experiment_model_name(f"Vanilla Qwen3.5-0.8B ({N_SHOT}-Shot)", TAG),
    few_shot_examples=few_shot_examples,
)
all_models_results = append_or_replace(all_models_results, fewshot_summary_df)
display(fewshot_results_df.head() if fewshot_results_df is not None else fewshot_summary_df)
display(headline_table(all_models_results, TASK))
del model_vanilla
free_gpu()

Loaded existing comparison table for task2 from 'bdatm-project/evaluation-results-task2' (21 model(s) already evaluated).
[task2] Using cached evaluation for 'Vanilla Qwen3.5-0.8B (Zero-Shot) [tok256]' -- skipping inference.
[task2] Using cached evaluation for 'Vanilla Qwen3.5-0.8B (3-Shot) [tok256]' -- skipping inference.


,sample_id,ground_truth,predicted,raw_output,exact_match,pred_san,em_extracted,multiline,legal,ambiguous,same_move,from_sq_match,to_sq_match,piece_match
0,sample_000000,Bxe4,Re7,Re7,0,Re7,0,0,0,0,0,0,0,0
1,sample_000001,Rxe6,Rc1,Rc1,0,Rc1,0,0,1,1,0,1,0,1
2,sample_000002,Rd1,Rc1,Rc1,0,Rc1,0,0,1,0,0,1,0,1
3,sample_000003,h6,Re7,Re7,0,Re7,0,0,0,0,0,0,0,0
4,sample_000004,Re1,Rc1,Rc1,0,Rc1,0,0,1,1,0,0,0,1


,model_name,setup,Exact Match,Same move (any notation),Legal move,EM raw output
0,Qwen + LoRA (File_wise - SFT),"legacy: pixel-tile reordering, 512px",0.6400,0.9775,0.9775,0.6400
1,Qwen + LoRA Fine-Tuning (from HF),"legacy: pixel-tile reordering, 512px",0.6200,0.9775,0.9775,0.6200
2,Qwen + LoRA (Raster - SFT) [tok256],"token reordering, 256px",0.6025,0.9375,0.9500,0.6025
3,Qwen + LoRA (Learned Reordering) [tok256],"token reordering, 256px",0.5775,0.8925,0.8975,0.5775
4,Qwen + LoRA (Spiral - SFT),"legacy: pixel-tile reordering, 512px",0.5775,0.9000,0.9150,0.5775
5,Qwen + LoRA (File_wise - SFT) [tok256],"token reordering, 256px",0.5475,0.8850,0.9050,0.5475
6,Qwen + LoRA (Zigzag - SFT),"legacy: pixel-tile reordering, 512px",0.5475,0.9200,0.9225,0.5475
7,Qwen + LoRA (Spiral_center_out - SFT) [tok256],"token reordering, 256px",0.5400,0.8250,0.8475,0.5400
8,Qwen + LoRA (Zigzag - SFT) [tok256],"token reordering, 256px",0.5350,0.8950,0.9075,0.5350
9,Qwen + LoRA (Learned Reordering),"legacy: pixel-tile reordering, 512px",0.5300,0.8700,0.8800,0.0000


## LoRA baseline (raster order)

Supervised LoRA fine-tuning with the native raster order (= rank-wise). Up to 10 epochs with early stopping on validation loss; the best epoch is kept, also across a Colab disconnect.

In [24]:
raster_repo = experiment_repo_id(TASK, "raster", TAG, hf_org_prefix)
trained = finetune_and_push_chessboard_model(
    "raster", dataset_task2, processor, load_base_model(), peft_config, TASK, hf_org_prefix=hf_org_prefix, tag=TAG,
)
del trained
free_gpu()

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]


TASK: TASK2 | STRATEGY: RASTER (token-level) -> bdatm-project/qwen-task2-raster-tok256-lora


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Found a resumable checkpoint on the Hub for 'bdatm-project/qwen-task2-raster-tok256-lora': /root/.cache/huggingface/hub/models--bdatm-project--qwen-task2-raster-tok256-lora/snapshots/a3f4ca73a61d58d07ef8529bb178aa011def455d/last-checkpoint
Resuming from /root/.cache/huggingface/hub/models--bdatm-project--qwen-task2-raster-tok256-lora/snapshots/a3f4ca73a61d58d07ef8529bb178aa011def455d/last-checkpoint
Training for 'bdatm-project/qwen-task2-raster-tok256-lora' already finished on the Hub -- skipping training and loading its final adapter.
Nothing new to push for bdatm-project/qwen-task2-raster-tok256-lora.


In [25]:
raster_model = PeftModel.from_pretrained(load_base_model(), raster_repo)
raster_results_df, raster_summary_df = evaluate_with_order(
    TASK, raster_model, processor, dataset_task2["test"], hf_org_prefix=hf_org_prefix, adapter_repo_id=raster_repo,
    model_name=experiment_model_name("Qwen + LoRA (Raster - SFT)", TAG),
)
all_models_results = append_or_replace(all_models_results, raster_summary_df)
display(headline_table(all_models_results, TASK))

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

[task2] Using cached evaluation for 'Qwen + LoRA (Raster - SFT) [tok256]' -- skipping inference.


,model_name,setup,Exact Match,Same move (any notation),Legal move,EM raw output
0,Qwen + LoRA (File_wise - SFT),"legacy: pixel-tile reordering, 512px",0.6400,0.9775,0.9775,0.6400
1,Qwen + LoRA Fine-Tuning (from HF),"legacy: pixel-tile reordering, 512px",0.6200,0.9775,0.9775,0.6200
2,Qwen + LoRA (Raster - SFT) [tok256],"token reordering, 256px",0.6025,0.9375,0.9500,0.6025
3,Qwen + LoRA (Spiral - SFT),"legacy: pixel-tile reordering, 512px",0.5775,0.9000,0.9150,0.5775
4,Qwen + LoRA (Learned Reordering) [tok256],"token reordering, 256px",0.5775,0.8925,0.8975,0.5775
5,Qwen + LoRA (Zigzag - SFT),"legacy: pixel-tile reordering, 512px",0.5475,0.9200,0.9225,0.5475
6,Qwen + LoRA (File_wise - SFT) [tok256],"token reordering, 256px",0.5475,0.8850,0.9050,0.5475
7,Qwen + LoRA (Spiral_center_out - SFT) [tok256],"token reordering, 256px",0.5400,0.8250,0.8475,0.5400
8,Qwen + LoRA (Zigzag - SFT) [tok256],"token reordering, 256px",0.5350,0.8950,0.9075,0.5350
9,Qwen + LoRA (Learned Reordering),"legacy: pixel-tile reordering, 512px",0.5300,0.8700,0.8800,0.0000


## Training-free reordering

The raster-trained LoRA model reads the *same* images with a different token order, with no retraining. Since the board is intact, this measures how much the model depends on the order it was trained with.

In [26]:
for strat in STRATEGIES:
    _, summary = evaluate_with_order(
        TASK, raster_model, processor, dataset_task2["test"], order=strat, hf_org_prefix=hf_org_prefix,
        adapter_repo_id=raster_repo, model_name=experiment_model_name(f"Qwen + LoRA Raster, read {strat} (TF)", TAG),
    )
    all_models_results = append_or_replace(all_models_results, summary)
display(headline_table(all_models_results, TASK))

[task2] Using cached evaluation for 'Qwen + LoRA Raster, read zigzag (TF) [tok256]' -- skipping inference.
[task2] Using cached evaluation for 'Qwen + LoRA Raster, read spiral_center_out (TF) [tok256]' -- skipping inference.
[task2] Using cached evaluation for 'Qwen + LoRA Raster, read file_wise (TF) [tok256]' -- skipping inference.
[task2] Using cached evaluation for 'Qwen + LoRA Raster, read hilbert (TF) [tok256]' -- skipping inference.


,model_name,setup,Exact Match,Same move (any notation),Legal move,EM raw output
0,Qwen + LoRA (File_wise - SFT),"legacy: pixel-tile reordering, 512px",0.6400,0.9775,0.9775,0.6400
1,Qwen + LoRA Fine-Tuning (from HF),"legacy: pixel-tile reordering, 512px",0.6200,0.9775,0.9775,0.6200
2,Qwen + LoRA (Raster - SFT) [tok256],"token reordering, 256px",0.6025,0.9375,0.9500,0.6025
3,Qwen + LoRA (Spiral - SFT),"legacy: pixel-tile reordering, 512px",0.5775,0.9000,0.9150,0.5775
4,Qwen + LoRA (Learned Reordering) [tok256],"token reordering, 256px",0.5775,0.8925,0.8975,0.5775
5,Qwen + LoRA (Zigzag - SFT),"legacy: pixel-tile reordering, 512px",0.5475,0.9200,0.9225,0.5475
6,Qwen + LoRA (File_wise - SFT) [tok256],"token reordering, 256px",0.5475,0.8850,0.9050,0.5475
7,Qwen + LoRA (Spiral_center_out - SFT) [tok256],"token reordering, 256px",0.5400,0.8250,0.8475,0.5400
8,Qwen + LoRA (Zigzag - SFT) [tok256],"token reordering, 256px",0.5350,0.8950,0.9075,0.5350
9,Qwen + LoRA (Learned Reordering),"legacy: pixel-tile reordering, 512px",0.5300,0.8700,0.8800,0.0000


## Fine-tuning with each scan order (SFT)

One LoRA run per strategy, trained and evaluated with that token order. Each run loads a fresh base model and frees it afterwards.

In [27]:
for strat in STRATEGIES:
    trained = finetune_and_push_chessboard_model(
        strat, dataset_task2, processor, load_base_model(), peft_config, TASK, hf_org_prefix=hf_org_prefix, tag=TAG,
    )
    del trained
    free_gpu()

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]


TASK: TASK2 | STRATEGY: ZIGZAG (token-level) -> bdatm-project/qwen-task2-zigzag-tok256-lora


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Found a resumable checkpoint on the Hub for 'bdatm-project/qwen-task2-zigzag-tok256-lora': /root/.cache/huggingface/hub/models--bdatm-project--qwen-task2-zigzag-tok256-lora/snapshots/73f8bc07cfe362bd97fd013891f6217570001048/last-checkpoint
Resuming from /root/.cache/huggingface/hub/models--bdatm-project--qwen-task2-zigzag-tok256-lora/snapshots/73f8bc07cfe362bd97fd013891f6217570001048/last-checkpoint
Training for 'bdatm-project/qwen-task2-zigzag-tok256-lora' already finished on the Hub -- skipping training and loading its final adapter.
Nothing new to push for bdatm-project/qwen-task2-zigzag-tok256-lora.


...

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]


TASK: TASK2 | STRATEGY: HILBERT (token-level) -> bdatm-project/qwen-task2-hilbert-tok256-lora


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Found a resumable checkpoint on the Hub for 'bdatm-project/qwen-task2-hilbert-tok256-lora': /root/.cache/huggingface/hub/models--bdatm-project--qwen-task2-hilbert-tok256-lora/snapshots/3fba05a7f921d84b115987497a21412635bb0981/last-checkpoint
Resuming from /root/.cache/huggingface/hub/models--bdatm-project--qwen-task2-hilbert-tok256-lora/snapshots/3fba05a7f921d84b115987497a21412635bb0981/last-checkpoint
Training for 'bdatm-project/qwen-task2-hilbert-tok256-lora' already finished on the Hub -- skipping training and loading its final adapter.
Nothing new to push for bdatm-project/qwen-task2-hilbert-tok256-lora.


In [28]:
for strat in STRATEGIES:
    repo = experiment_repo_id(TASK, strat, TAG, hf_org_prefix)
    strat_model = PeftModel.from_pretrained(load_base_model(), repo)
    _, summary = evaluate_with_order(
        TASK, strat_model, processor, dataset_task2["test"], order=strat, hf_org_prefix=hf_org_prefix, adapter_repo_id=repo,
        model_name=experiment_model_name(f"Qwen + LoRA ({strat.capitalize()} - SFT)", TAG),
    )
    all_models_results = append_or_replace(all_models_results, summary)
    del strat_model
    free_gpu()
display(headline_table(all_models_results, TASK))

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

[task2] Using cached evaluation for 'Qwen + LoRA (Zigzag - SFT) [tok256]' -- skipping inference.


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

[task2] Using cached evaluation for 'Qwen + LoRA (Spiral_center_out - SFT) [tok256]' -- skipping inference.


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

[task2] Using cached evaluation for 'Qwen + LoRA (File_wise - SFT) [tok256]' -- skipping inference.


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

[task2] Using cached evaluation for 'Qwen + LoRA (Hilbert - SFT) [tok256]' -- skipping inference.


,model_name,setup,Exact Match,Same move (any notation),Legal move,EM raw output
0,Qwen + LoRA (File_wise - SFT),"legacy: pixel-tile reordering, 512px",0.6400,0.9775,0.9775,0.6400
1,Qwen + LoRA Fine-Tuning (from HF),"legacy: pixel-tile reordering, 512px",0.6200,0.9775,0.9775,0.6200
2,Qwen + LoRA (Raster - SFT) [tok256],"token reordering, 256px",0.6025,0.9375,0.9500,0.6025
3,Qwen + LoRA (Spiral - SFT),"legacy: pixel-tile reordering, 512px",0.5775,0.9000,0.9150,0.5775
4,Qwen + LoRA (Learned Reordering) [tok256],"token reordering, 256px",0.5775,0.8925,0.8975,0.5775
5,Qwen + LoRA (Zigzag - SFT),"legacy: pixel-tile reordering, 512px",0.5475,0.9200,0.9225,0.5475
6,Qwen + LoRA (File_wise - SFT) [tok256],"token reordering, 256px",0.5475,0.8850,0.9050,0.5475
7,Qwen + LoRA (Spiral_center_out - SFT) [tok256],"token reordering, 256px",0.5400,0.8250,0.8475,0.5400
8,Qwen + LoRA (Zigzag - SFT) [tok256],"token reordering, 256px",0.5350,0.8950,0.9075,0.5350
9,Qwen + LoRA (Learned Reordering),"legacy: pixel-tile reordering, 512px",0.5300,0.8700,0.8800,0.0000


## Learned reordering (REOrder)

REOrder's Plackett-Luce policy (`external/REOrder`) over the 64 squares, trained jointly with LoRA by REINFORCE (`src/training/learned_reordering.py`): one warm-up epoch in raster order, then the policy samples orders with a Gumbel temperature annealed 0.2 -> 0. Reward = minus the answer's cross-entropy; two orders are scored on each sample and compared with each other, so puzzle difficulty cancels out. Up to 10 epochs with early stopping on validation loss (patience 2); the best epoch is kept, together with the order it was validated with (the policy's greedy order, or raster if the warm-up epoch was best). Resumable from Hub checkpoints.

In [29]:
learned_model, learned_policy, learned_repo, learned_order = train_with_learned_reordering(
    dataset_task2, processor, load_base_model(), peft_config, task=TASK, hf_org_prefix=hf_org_prefix, tag=TAG,
    num_train_epochs=10, policy_start_after_epochs=1,
)
print("Order used for evaluation (reading step of each square):")
print(describe_order(learned_order))
print("\nPolicy's greedy order:")
print(describe_order(greedy_order(learned_policy)))

_, learned_summary = evaluate_with_order(
    TASK, learned_model, processor, dataset_task2["test"], order=learned_order, hf_org_prefix=hf_org_prefix,
    adapter_repo_id=learned_repo, model_name=experiment_model_name("Qwen + LoRA (Learned Reordering)", TAG),
)
all_models_results = append_or_replace(all_models_results, learned_summary)
display(headline_table(all_models_results, TASK))
del learned_model
free_gpu()

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

[task2] Resuming learned reordering at epoch 4, row 0 (optimizer step 1600).


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

  ...k256-lora/final/policy.pt: 100%|##########| 2.08kB / 2.08kB            

Validating                    :           |   0%            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


Order used for evaluation (reading step of each square):
8  1  0  3  2  6  7  5  4
7  8  9 12 10 11 13 17 14
6 15 19 18 16 20 22 21 26
5 25 24 27 23 28 29 30 31
4 33 32 34 36 35 37 40 39
3 38 42 41 43 45 44 46 47
2 49 48 50 51 53 52 55 54
1 56 61 57 59 58 60 63 62
   a  b  c  d  e  f  g  h

Policy's greedy order:
8  1  0  3  2  6  7  5  4
7  8  9 12 10 11 13 17 14
6 15 19 18 16 20 22 21 26
5 25 24 27 23 28 29 30 31
4 33 32 34 36 35 37 40 39
3 38 42 41 43 45 44 46 47
2 49 48 50 51 53 52 55 54
1 56 61 57 59 58 60 63 62
   a  b  c  d  e  f  g  h
[task2] Using cached evaluation for 'Qwen + LoRA (Learned Reordering) [tok256]' -- skipping inference.


,model_name,setup,Exact Match,Same move (any notation),Legal move,EM raw output
0,Qwen + LoRA (File_wise - SFT),"legacy: pixel-tile reordering, 512px",0.6400,0.9775,0.9775,0.6400
1,Qwen + LoRA Fine-Tuning (from HF),"legacy: pixel-tile reordering, 512px",0.6200,0.9775,0.9775,0.6200
2,Qwen + LoRA (Raster - SFT) [tok256],"token reordering, 256px",0.6025,0.9375,0.9500,0.6025
3,Qwen + LoRA (Learned Reordering) [tok256],"token reordering, 256px",0.5775,0.8925,0.8975,0.5775
4,Qwen + LoRA (Spiral - SFT),"legacy: pixel-tile reordering, 512px",0.5775,0.9000,0.9150,0.5775
5,Qwen + LoRA (File_wise - SFT) [tok256],"token reordering, 256px",0.5475,0.8850,0.9050,0.5475
6,Qwen + LoRA (Zigzag - SFT),"legacy: pixel-tile reordering, 512px",0.5475,0.9200,0.9225,0.5475
7,Qwen + LoRA (Spiral_center_out - SFT) [tok256],"token reordering, 256px",0.5400,0.8250,0.8475,0.5400
8,Qwen + LoRA (Zigzag - SFT) [tok256],"token reordering, 256px",0.5350,0.8950,0.9075,0.5350
9,Qwen + LoRA (Learned Reordering),"legacy: pixel-tile reordering, 512px",0.5300,0.8700,0.8800,0.0000


## Resolution ablation

Same raster LoRA recipe at 512px (legacy run: 2x2 tokens per square) vs 256px (1 token per square).

In [30]:
LEGACY_RASTER_512 = "Qwen + LoRA Fine-Tuning (from HF)"   # raster LoRA of the first runs, 512px images
rows = all_models_results["model_name"].isin([LEGACY_RASTER_512, experiment_model_name("Qwen + LoRA (Raster - SFT)", TAG)])
display(headline_table(all_models_results[rows], TASK))

,model_name,setup,Exact Match,Same move (any notation),Legal move,EM raw output
0,Qwen + LoRA Fine-Tuning (from HF),"legacy: pixel-tile reordering, 512px",0.6200,0.9775,0.9775,0.6200
1,Qwen + LoRA (Raster - SFT) [tok256],"token reordering, 256px",0.6025,0.9375,0.9500,0.6025


## Results

Diagnostic columns are recomputed from the cached per-sample predictions (no inference), then the table is pushed to the Hub and reloaded from there. `headline_table` shows the main metrics first.

In [31]:
all_models_results = backfill_cached_metrics(
    all_models_results, TASK, fens=dataset_task2["test"]["fen"], hf_org_prefix=hf_org_prefix,
    drop_models=[  # all-zero rows left by an early, broken Task 2 run (no per-sample data)
        "Vanilla Qwen3.5-0.8B (Zero-Shot)", "Qwen + LoRA Task 2 (Raster - SFT)", "Qwen + LoRA Task 2 (Zigzag - SFT)",
        "Qwen + LoRA Task 2 (Spiral - SFT)", "Qwen + LoRA Task 2 (File_wise - SFT)",
    ]
)
results_repo = f"{hf_org_prefix}/evaluation-results-{TASK}"
push_results_table(results_repo, all_models_results, commit_message=f"{TASK}: comparison table")
display(headline_table(load_results_table(results_repo), TASK))

[task2] recomputed diagnostics for 21 row(s) from per-sample CSVs


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 9.49kB / 9.49kB            

Validating                    :           |   0%            

No files have been modified since last commit. Skipping to prevent empty commit.


,model_name,setup,Exact Match,Same move (any notation),Legal move,EM raw output
0,Qwen + LoRA (File_wise - SFT),"legacy: pixel-tile reordering, 512px",0.6400,0.9775,0.9775,0.6400
1,Qwen + LoRA Fine-Tuning (from HF),"legacy: pixel-tile reordering, 512px",0.6200,0.9775,0.9775,0.6200
2,Qwen + LoRA (Raster - SFT) [tok256],"token reordering, 256px",0.6025,0.9375,0.9500,0.6025
3,Qwen + LoRA (Learned Reordering) [tok256],"token reordering, 256px",0.5775,0.8925,0.8975,0.5775
4,Qwen + LoRA (Spiral - SFT),"legacy: pixel-tile reordering, 512px",0.5775,0.9000,0.9150,0.5775
5,Qwen + LoRA (File_wise - SFT) [tok256],"token reordering, 256px",0.5475,0.8850,0.9050,0.5475
6,Qwen + LoRA (Zigzag - SFT),"legacy: pixel-tile reordering, 512px",0.5475,0.9200,0.9225,0.5475
7,Qwen + LoRA (Spiral_center_out - SFT) [tok256],"token reordering, 256px",0.5400,0.8250,0.8475,0.5400
8,Qwen + LoRA (Zigzag - SFT) [tok256],"token reordering, 256px",0.5350,0.8950,0.9075,0.5350
9,Qwen + LoRA (Learned Reordering),"legacy: pixel-tile reordering, 512px",0.5300,0.8700,0.8800,0.0000


## Diagnostics

`training_health_report` per run: eval-loss history, discontinuities, recipe markers, best vs published epoch.

In [32]:
if CONFIG["colab"]:
    from google.colab import drive
    drive.mount("/content/drive")   # save_diagnostics writes here, surviving a runtime reset

health = {r: training_health_report(r) for r in [experiment_repo_id(TASK, s, TAG, hf_org_prefix) for s in ["raster", *STRATEGIES, "learned"]]}
health_df = pd.DataFrame([
    {"repo_id": r, "n_problems": len(h["problems"]), "problems": " | ".join(h["problems"])} for r, h in health.items()
])
display(health_df)
save_diagnostics(health_df, f"training_health_{TAG}", TASK)

Mounted at /content/drive
=== bdatm-project/qwen-task2-raster-tok256-lora ===
  eval_loss by epoch: 1:0.324, 2:0.253, 3:0.179, 4:0.156, 5:0.141, 6:0.135, 7:0.168, 8:0.185
  train loss first/last: 1.1440773010253906 / 0.03261309266090393
  - no problems detected
=== bdatm-project/qwen-task2-zigzag-tok256-lora ===
  eval_loss by epoch: 1:0.381, 2:0.277, 3:0.191, 4:0.171, 5:0.153, 6:0.148, 7:0.168, 8:0.173
  train loss first/last: 1.426084041595459 / 0.03569161891937256
  - no problems detected
=== bdatm-project/qwen-task2-spiral_center_out-tok256-lora ===
  eval_loss by epoch: 1:0.365, 2:0.266, 3:0.240, 4:0.182, 5:0.168, 6:0.187, 7:0.189
  train loss first/last: 1.1298017501831055 / 0.05417528748512268
  - no problems detected
=== bdatm-project/qwen-task2-file_wise-tok256-lora ===
  eval_loss by epoch: 1:0.375, 2:0.285, 3:0.226, 4:0.168, 5:0.168, 6:0.158, 7:0.167, 8:0.218
  train loss first/last: 0.9773617744445801 / 0.02673819661140442
  - no problems detected
=== bdatm-project/qwen-tas

,repo_id,n_problems,problems
0,bdatm-project/qwen-task2-raster-tok256-lora,0,
1,bdatm-project/qwen-task2-zigzag-tok256-lora,0,
2,bdatm-project/qwen-task2-spiral_center_out-tok...,0,
3,bdatm-project/qwen-task2-file_wise-tok256-lora,0,
4,bdatm-project/qwen-task2-hilbert-tok256-lora,0,
5,bdatm-project/qwen-task2-learned-tok256-lora,0,


Saved diagnostics to /content/drive/MyDrive/bdatm_diagnostics/task2/training_health_tok256.csv


PosixPath('/content/drive/MyDrive/bdatm_diagnostics/task2/training_health_tok256.csv')

Qualitative error analysis of the raster LoRA model: one failure mode per prediction (`same move, notation differs` = right move, written differently).

In [33]:
analyzed = (raster_results_df if "same_move" in raster_results_df.columns
            else analyze_san_predictions(raster_results_df, dataset_task2["test"]["fen"]))
display(san_error_breakdown(analyzed))
save_diagnostics(analyzed, f"raster_lora_per_sample_{TAG}", TASK)

,count
exact,241
"same move, notation differs",121
illegal / unparseable,20
"same move, under-disambiguated",13
"right piece moved, wrong destination",5


Saved diagnostics to /content/drive/MyDrive/bdatm_diagnostics/task2/raster_lora_per_sample_tok256.csv


PosixPath('/content/drive/MyDrive/bdatm_diagnostics/task2/raster_lora_per_sample_tok256.csv')